<a href="https://colab.research.google.com/github/pia-francesca/ema/blob/main/examples/Pla2g2/emmaemb_pla2g2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EmmaEmb: Comparative Analysis of Embedding Spaces

Welcome to the example Colab notebook for EmmaEmb, a Python library for analyzing and comparing embedding spaces in molecular biology. EmmaEmb provides tools to explore how different embedding models capture biological information, enabling insights into feature similarities, differences, and relationships across embeddings.

Link to GitHub: https://github.com/broadinstitute/EmmaEmb


### Notebook content

This notebook demonstrates key functionalities of EmmaEmb, including:

1. [Initialising the Emma object](#section-one)

2. [Adding embedding spaces](#section-two)

3. [Embedding space diagnostics](#section-three)

4. [Feature distribution across spaces](#section-four)

5. [Pairwise space comparison](#section-five)

![EmmaEmb Overview](https://raw.githubusercontent.com/broadinstitute/EmmaEmb/main/images/emma_overview.jpg)


## 0. Loading dependencies and data

Information of the data and embedding models can be found here: https://github.com/broadinstitute/EmmaEmb/tree/main/examples/Pla2g2

In [ ]:
#@title Install dependencies
%pip install --upgrade emmaemb

In [ ]:
#@title Download example data from EmmaEmb repository

import requests
import pandas as pd
import os

# download embeddings

models = ["ESMC", "ProtT5"]
embedding_url_dir = "https://raw.githubusercontent.com/broadinstitute/EmmaEmb/main/examples/Pla2g2/embeddings/"

headers = {"User-Agent": "Mozilla/5.0"}
csv_url = "https://raw.githubusercontent.com/broadinstitute/EmmaEmb/main/examples/Pla2g2/Pla2g2_features.csv"

csv_filename = "Pla2g2_features.csv"
csv_response = requests.get(csv_url, headers=headers)
if csv_response.status_code == 200:
    with open(csv_filename, "wb") as f:
        f.write(csv_response.content)
else:
    print(f"Failed to download {csv_filename}")


df_pla2g2 = pd.read_csv(csv_filename)
proteins = df_pla2g2['identifier'].values

# now for each model download embedding files for each protein
for model in models:
  model_dir = f"embeddings/{model}"
  os.makedirs(model_dir, exist_ok=True)

  for protein in proteins:
    file_path = os.path.join(model_dir, f"{protein}.npy")

    # Check if file already exists
    if os.path.exists(file_path):
        continue

    url = f"{embedding_url_dir}{model}/{protein}.npy"
    response = requests.get(url, headers=headers)

    if response.status_code == 200:
      # store in embeddings/model/protein-id.npy
      with open("embeddings/" + model + "/" + protein + ".npy", "wb") as f:
        f.write(response.content)
    else:
      print(f"Failed to download {url}")


print("All download of feature data complete.")

<a name="section-one"></a>
## 1. Initialising the Emma object

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from emmaemb import Emma

colors = {
    "ProtT5":  "#4D4D4D",
    "ESMC":    "#303496",
}

### Feature data table

Loading feature data. The first column includes the identifiers of the samples, in this case proteins. The remaining columns contain meta data on each sample.

In [ ]:
df_pla2f2 = pd.read_csv("Pla2g2_features.csv")
print(df_pla2f2.shape)
df_pla2f2.head()

Initialising Emma object with the feature data in format of a pandas df. The datatype stored in each column of the feature data is detected. Only cateorical data will be available for downstream analysis with the Emma library.

Note: Quantitative features can be binned to allow analysis with EmmaEmb.

In [ ]:
# initiate Emma object with the metadata

emma = Emma(df_pla2f2)

<a name="section-two"></a>
## 2. Adding embedding spaces

Adding embedding spaces. Embedding spaces can be added one by one. Either by

- providing a link to a directory which stores the embeddings in individual files with the identifiers from the feature table or

- by providing a numpy array which includes the embeddings in each row and in the same order as in the feature data table.


Multiple embedding spaces can be added. Dimensions of the embeddings do not have to be the same across embedding spaces. Embedding spaces can be removed using the `remove_emb_space(emb_space_name: str)` function.

In [ ]:
embedding_dir = "embeddings/"
models = ["ProtT5", "ESMC"]

In [ ]:
for model_name in models:
    emma.add_emb_space(
        embeddings_source=embedding_dir + model_name,
        emb_space_name=model_name,
    )

### Visualization of embedding spaces using dimensionality reduction techniques

The `plot_emb_space` function visualizes the embeddings of a specified embedding space in 2D using dimensionality reduction techniques such as PCA, t-SNE, or UMAP. It takes an Emma instance containing multiple embedding spaces and projects the selected space into two dimensions, optionally normalizing the data beforehand. The resulting scatter plot can be colored based on metadata attributes, allowing for an intuitive exploration of patterns within the embedding space.

In [ ]:
# visualise reduced embedding space
from emmaemb.visualization import plot_emb_space

fig_pca = plot_emb_space(emma=emma,
                         emb_space="ProtT5",
                         method="PCA",
                         color_by="enzyme_class",
                         normalise=True)
fig_pca.show()

<a name="section-three"></a>
## 3. Embedding space diagnostics

Before comparing models, this section runs a 6-step diagnostic protocol that assesses the geometric quality of each embedding space and produces a ranked, confidence-flagged result.

| Step | Purpose |
|---|---|
| **1 — Anisotropy assessment** | Detect directional bias; apply mean-centering where needed and select the appropriate distance metric |
| **2 — Hubness assessment** | Identify whether a small number of hub points dominate k-nearest-neighbour lists |
| **3 — Class distribution and k range** | Count samples per class, derive n_min as a hard upper bound on k, compute random baselines |
| **4 — Parameter sensitivity analysis** | Measure KNN alignment across the full k range and all distance metrics |
| **5 — Class imbalance robustness** | Progressively downsample the majority class to verify imbalance is not driving rankings |
| **6 — Feature noise robustness** | Progressively perturb feature labels to quantify how tight class structure is |

> **Disclaimer:** The decision thresholds used throughout this workflow (e.g. RHI > 0.3 for high hubness, average cosine similarity > 0.3 for anisotropy, noise crossing point > 0.3 for robustness) are chosen based on empirical observation rather than established literature benchmarks. They are intended as a reasonable starting point for interpretation. Further empirical evaluation on a broader range of datasets and tasks will be needed to validate or refine these thresholds.

### 3.1 Anisotropy

An isotropic embedding space is one where vectors point in many different directions. High anisotropy — where most embeddings cluster along a narrow cone — can inflate cosine similarities and distort nearest-neighbor structure.

`get_anisotropy_diagnostics` measures the average pairwise cosine similarity between random vector pairs. Values close to 0 indicate near-isotropic geometry; values approaching 1 indicate severe anisotropy. The diagnostic also reports whether mean-centering is likely to help.

In [ ]:
from emmaemb.functions import get_anisotropy_diagnostics

feature          = "enzyme_class"
distance_metrics = ["cosine", "cityblock"]

aniso = get_anisotropy_diagnostics(emma, n_pairs=10_000, seed=42)
print(aniso)

In [ ]:
df_aniso = aniso.data["summary"]

df_orig = df_aniso[["Embedding", "AvgPairwiseCosineSim"]].copy()
df_orig["Type"] = "Original"
df_orig = df_orig.rename(columns={"AvgPairwiseCosineSim": "Score"})

df_mc = df_aniso[["Embedding", "AvgPairwiseCosineSim_MC"]].dropna(subset=["AvgPairwiseCosineSim_MC"]).copy()
df_mc["Type"] = "Mean-centred"
df_mc = df_mc.rename(columns={"AvgPairwiseCosineSim_MC": "Score"})

df_plot = pd.concat([df_orig, df_mc], ignore_index=True)
type_colors = {"Original": "#303496", "Mean-centred": "#B4063D"}

fig = px.bar(
    df_plot, x="Embedding", y="Score", color="Type", barmode="group",
    text=df_plot["Score"].map("{:.3f}".format),
    title="Anisotropy — average pairwise cosine similarity",
    labels={"Score": "Avg cosine similarity", "Embedding": "", "Type": ""},
    template="plotly_white",
    color_discrete_map=type_colors,
)
fig.update_traces(textposition="outside")
fig.update_layout(
    yaxis=dict(range=[0, df_plot["Score"].max() * 1.2]),
    font=dict(family="Arial", size=14),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig.add_hline(y=0, line_dash="dot", line_color="grey",
              annotation_text="isotropic baseline (0)", annotation_position="bottom right")
fig.update_xaxes(showline=True, linecolor="black", linewidth=2, showgrid=False)
fig.update_yaxes(showline=True, linecolor="black", linewidth=2, showgrid=False)
fig.show()

In [ ]:
var_dict = aniso.data["variance_per_dim"]

fig = go.Figure()
for model, var in var_dict.items():
    sorted_var = np.sort(var)[::-1]
    cumvar = np.cumsum(sorted_var) / sorted_var.sum()
    n90 = int(np.searchsorted(cumvar, 0.90)) + 1
    print(f"{model}: {n90} dims capture 90% of variance (out of {len(var)})")
    fig.add_trace(go.Scatter(
        x=np.arange(len(cumvar)), y=cumvar,
        mode="lines", name=model,
        line=dict(color=colors[model], width=2),
    ))

fig.add_hline(y=0.90, line_dash="dash", line_color="grey",
              annotation_text="90%", annotation_position="right")
fig.update_layout(
    title="Cumulative explained variance",
    xaxis_title="Number of dimensions",
    yaxis_title="Cumulative variance fraction",
    yaxis=dict(range=[0, 1.05]),
    template="plotly_white",
    font=dict(family="Arial", size=14),
)
fig.update_xaxes(showline=True, linecolor="black", linewidth=2, showgrid=False)
fig.update_yaxes(showline=True, linecolor="black", linewidth=2, showgrid=False)
fig.show()

Mean-centering subtracts the per-dimension mean, shifting the embedding cloud to the origin. The cell below automatically detects which embedding spaces benefit from centering (those where it resolves the anisotropy score) and applies it in-place. All cached pairwise distances are cleared and recomputed for all metrics.

In [ ]:
anisotropic_spaces = (
    aniso.data["summary"]
    .dropna(subset=["AnisotropyScore_MC"])["Embedding"]
    .tolist()
)
if anisotropic_spaces:
    print(f"Applying mean-centering to: {anisotropic_spaces}")
    emma.mean_center(emb_spaces=anisotropic_spaces)
else:
    print("No mean-centering applied (all spaces are isotropic).")

for metric in distance_metrics:
    for model in models:
        emma.calculate_pairwise_distances(model, metric)
        print(f"  {model} / {metric} done")

### 3.2 Hubness

Hubness is a phenomenon in high-dimensional spaces where a small number of points become the nearest neighbors of a disproportionately large number of others — not because they are genuinely similar to everything, but as a mathematical consequence of high dimensionality. Hub points can inflate KNN-based alignment scores and bias downstream analyses.

`get_hubness_diagnostics` returns the Robin Hood index (a summary of how unequally k-occurrences are distributed across samples) and the per-sample k-occurrence distribution.

In [ ]:
from emmaemb.functions import get_hubness_diagnostics

k_hub = 10
hub = get_hubness_diagnostics(emma, k=k_hub, metric="cosine")
print(hub)

In [ ]:
rhi_df = hub.data["rhi"]

fig = px.bar(
    rhi_df, x="Embedding", y="RobinHoodIndex",
    color="Embedding",
    text=rhi_df["RobinHoodIndex"].map("{:.3f}".format),
    title="Robin Hood index (k=10, cosine)",
    labels={"RobinHoodIndex": "Robin Hood index", "Embedding": ""},
    template="plotly_white",
    color_discrete_map=colors,
)
fig.update_traces(textposition="outside")
fig.update_layout(
    yaxis=dict(range=[0, rhi_df["RobinHoodIndex"].max() * 1.35]),
    showlegend=False, font=dict(family="Arial", size=14),
)
fig.update_xaxes(showline=True, linecolor="black", linewidth=2, showgrid=False)
fig.update_yaxes(showline=True, linecolor="black", linewidth=2, showgrid=False)
fig.show()

In [ ]:
kocc_df = hub.data["k_occurrence"]

fig = px.histogram(
    kocc_df, x="KOccurrence", color="Embedding",
    barmode="overlay", nbins=20, opacity=0.65,
    title=f"k-occurrence distribution (k={k_hub}, cosine)",
    labels={"KOccurrence": "k-occurrence count"},
    template="plotly_white",
    color_discrete_map=colors,
)
fig.add_vline(x=k_hub, line_dash="dot", line_color="grey",
              annotation_text=f"expected mean = {k_hub} (i.e. k)",
              annotation_position="top right")
fig.update_layout(font=dict(family="Arial", size=14))
fig.update_xaxes(showline=True, linecolor="black", linewidth=2, showgrid=False)
fig.update_yaxes(showline=True, linecolor="black", linewidth=2, showgrid=False)
fig.show()

In [ ]:
# Steps 1–2: geometry summary
df_aniso_s = aniso.data["summary"]
rhi_df_s = hub.data["rhi"].set_index("Embedding")
rhi_max_ = float(hub.data["rhi"]["RobinHoodIndex"].max())

print("Recommended distance metric:")
for _, row in df_aniso_s.iterrows():
    cos = row["AvgPairwiseCosineSim"]
    mc  = row.get("MeanCenteringHelps", False)
    if cos > 0.3:
        rec = "cosine  ⚠  high anisotropy"
    elif cos > 0.1:
        rec = "cosine  (mild anisotropy)"
    else:
        rec = "cosine / euclidean / manhattan (isotropic)"
    mc_note = "  → mean-centering applied" if mc else ""
    print(f"  {row['Embedding']:12s}  avg cosine sim = {cos:.3f}  → {rec}{mc_note}")

print()
print("Hubness flags:")
for emb, row in rhi_df_s.iterrows():
    rhi = row["RobinHoodIndex"]
    flag = "⚠ high — use smaller k" if rhi > 0.3 else "OK"
    print(f"  {emb:12s}  RHI = {rhi:.3f}  → {flag}")

---
## Step 3 — Class distribution and k range

Count samples per class, identify **n_min** (hard upper bound on k), compute the imbalance ratio, and derive per-class random baselines.

In [ ]:
class_counts = emma.metadata[feature].value_counts().reset_index()
class_counts.columns = [feature, "count"]

fig = px.bar(
    class_counts, x=feature, y="count",
    text="count",
    title=f"Class distribution — {feature}",
    template="plotly_white",
    color=feature,
)
fig.update_traces(textposition="outside")
fig.update_layout(
    yaxis_title="Number of sequences",
    font=dict(family="Arial", size=13),
    xaxis_tickangle=-35,
    showlegend=False,
)
fig.update_xaxes(showline=True, linecolor="black", linewidth=2, showgrid=False)
fig.update_yaxes(showline=True, linecolor="black", linewidth=2, showgrid=False)
fig.show()

N_total   = int(class_counts["count"].sum())
n_min     = int(class_counts["count"].min())
n_max     = int(class_counts["count"].max())
imbalance = n_max / n_min
n_classes = int(class_counts[feature].nunique())

print(f"N = {N_total}, classes = {n_classes}, n_min = {n_min}, n_max = {n_max}")
print(f"Imbalance ratio: {imbalance:.1f}x  ('⚠ flag — run robustness sweep in Step 5' if imbalance > 3 else '✓ moderate')")
print(f"Uniform random baseline: 1/{n_classes} = {1/n_classes:.3f}")

In [ ]:
N_total  = len(emma.metadata)
n_min    = int(emma.metadata[feature].value_counts().min())
rhi_max_ = hub.data["rhi"]["RobinHoodIndex"].max()

k_hard   = n_min - 1
k_min_v  = min(5, k_hard)
k_sqrt   = int(np.sqrt(N_total))
k_pct    = max(1, int(0.05 * N_total))

if rhi_max_ > 0.3:
    k_ceiling = max(k_min_v, min(k_hard, k_pct // 2))
    print(f"⚠  High hubness (max RHI = {rhi_max_:.3f}): applying conservative k ceiling")
else:
    k_ceiling = max(k_min_v, min(k_hard, max(k_sqrt, k_pct)))

if k_ceiling < 20:
    k_values = [k for k in [3, 5, 10, 15, 20] if k_min_v <= k <= k_hard]
else:
    k_values = [k for k in [1, 5, 10, 20, 30, 50, 100] if k_min_v <= k <= k_ceiling]
if not k_values:
    k_values = [k_min_v]

print(f"N = {N_total},  n_min = {n_min}")
print(f"k hard upper bound (< n_min):    {k_hard}")
print(f"k ceiling applied:               {k_ceiling}")
print(f"k values to sweep:               {k_values}")
print(f"Distance metrics:                {distance_metrics}")

In [ ]:
# Edit below to override k_values if needed
# k_values = [10, 20, 30, 50]
print(f"k_values: {k_values}")

<a name="section-four"></a>
## 4. Feature distribution across spaces

The `calculate_pairwise_distances` method computes pairwise distances between samples in an embedding space and caches the k-nearest neighbor ranks for downstream analyses. Supported distance metrics include Euclidean, Manhattan, Cosine, and several normalized variants. Distances are only computed once — subsequent calls for the same space and metric return immediately.

In [ ]:
# Pairwise distances were computed in the mean-centering cell above.
# Verify the cache is populated:
print("Cached distances:")
for model in models:
    for metric in distance_metrics:
        ranks = emma.emb.get(model, {}).get("ranks", {})
        status = "✓" if metric in ranks else "✗"
        print(f"  {status} {model} / {metric}")

#### 4.1 KNN feature alignment scores

### 4.3 KNN alignment across k

The choice of k affects alignment scores. `plot_knn_alignment_across_k` sweeps k across a range and plots the mean alignment score for each embedding space. Stable rankings across k indicate robust results; crossings or inversions suggest parameter sensitivity and should be reported.

In [ ]:
from emmaemb.visualization import plot_knn_alignment_across_k

fig_knn_k, df_knn = plot_knn_alignment_across_k(
    emma=emma,
    feature=feature,
    k_values=k_values,
    metrics=distance_metrics,
    show_random_baselines=True,
    elbow_detection=True,
    return_data=True,
)
fig_knn_k.update_layout(height=500, width=900)
fig_knn_k.show()

# Step 4: parameter sensitivity
top_per_k = (
    df_knn.groupby(["distance_metric", "k"])
    .apply(lambda g: g.loc[g["Fraction"].idxmax(), "Embedding"])
    .rename("top_model")
    .reset_index()
)

top_per_metric = {
    metric: top_per_k[top_per_k["distance_metric"] == metric]["top_model"].mode()[0]
    for metric in distance_metrics
    if metric in top_per_k["distance_metric"].values
}

all_k_stable = len(top_per_k["top_model"].unique()) == 1
metrics_agree = len(set(top_per_metric.values())) == 1

if all_k_stable and metrics_agree:
    stability_flag = "STABLE"
    primary_model = top_per_k["top_model"].mode()[0]
    stability_rec = f"Rankings consistent across k and metrics. Recommend {primary_model}."
elif all_k_stable and not metrics_agree:
    stability_flag = "METRIC-SENSITIVE"
    primary_model = top_per_metric.get("cosine", list(top_per_metric.values())[0])
    stability_rec = f"Stable across k but metric-dependent. Primary (cosine): {primary_model}."
else:
    stability_flag = "PARAMETER-SENSITIVE"
    primary_model = top_per_k["top_model"].mode()[0]
    stability_rec = "Rankings invert across k. Results are parameter-sensitive."

print(f"\nStep 4 — Stability flag: {stability_flag}")
print(f"  {stability_rec}")
print(f"  Top model per metric: {top_per_metric}")

### 4.4 Within/between class distance distributions

`plot_within_between_distributions` visualises the overlap between within-class and between-class pairwise distances for a given embedding space. A clean separation indicates that the embedding geometry reflects the class structure well; large overlap suggests poor class separation for the chosen metric.

In [ ]:
from emmaemb.visualization import plot_within_between_distributions

fig_wb = plot_within_between_distributions(
    emma=emma,
    emb_space="ProtT5",
    metric="cityblock",
    feature="enzyme_class",
)
fig_wb.update_layout(height=500, width=700)
fig_wb.show()

The `plot_knn_alignment_across_embedding_spaces` function visualizes k-nearest neighbor (KNN) feature alignment scores for a specified feature across multiple embedding spaces.
It computes what fraction of the KNN embeddings are labelled with the same label for the selected feature.
The scores are calculated for each embedding in each embedding space.
Pairwise distances need to be pre-computed for each embedding space and each distance metric by calling `calculate_pairwise_distances` beforehand (see above).
The function produces a box plot and allows customization of the embedding space order and plot color.

In [ ]:
# KNN ALIGNMENT SCORES
from emmaemb.visualization import plot_knn_alignment_across_embedding_spaces

fig_alignment_scores = plot_knn_alignment_across_embedding_spaces(
    emma, feature="enzyme_class", k=10, metric="cityblock"
)
fig_alignment_scores.update_layout(height=600, width=500)
fig_alignment_scores.show()

The KNN feature alignment scores can also be aggregated. The `plot_knn_alignment_across_classes` shows a heatmap of the mean value of the KNN feature alignment scores stratified by embedding space and feature class.

### 5.3 Robustness to class imbalance and label noise

Alignment scores can be affected by class imbalance (large classes dominate neighbors) and label quality. The following two plots assess robustness to these factors, helping distinguish genuine geometric class structure from frequency-driven or noise-sensitive artefacts.

#### 5.3a Class imbalance

`plot_knn_alignment_vs_class_balance` progressively downsamples the majority class toward the size of the smallest class and repeats KNN alignment. Stable rankings across this sweep indicate that the result is not driven by class frequency.

In [ ]:
from emmaemb.visualization import plot_knn_alignment_vs_class_balance

fig_balance, df_balance = plot_knn_alignment_vs_class_balance(
    emma=emma,
    feature=feature,
    emb_spaces=models,
    k_values=k_values,
    metrics=distance_metrics,
    n_balance_steps=6,
    seed=42,
    show_random_baselines=True,
    return_data=True,
)
fig_balance.update_layout(height=500, width=900)
fig_balance.show()

# Step 5: class imbalance robustness
metric_display = {"cosine": "Cosine distance", "cityblock": "Manhattan distance", "euclidean": "Euclidean distance"}
k_ref = k_values[len(k_values) // 2]
primary_metric_label = metric_display.get(distance_metrics[0], distance_metrics[0])

df_b = df_balance[
    (df_balance["k"] == k_ref) & (df_balance["Metric"] == primary_metric_label)
].copy()

if not df_b.empty:
    cap_min = df_b["Max samples per class"].min()
    cap_max = df_b["Max samples per class"].max()
    top_max_cap = df_b.loc[df_b[df_b["Max samples per class"] == cap_max]["Fraction"].idxmax(), "Embedding"]
    top_min_cap = df_b.loc[df_b[df_b["Max samples per class"] == cap_min]["Fraction"].idxmax(), "Embedding"]

    if top_max_cap == top_min_cap:
        balance_flag = "CONSISTENT"
        balance_note = f"Top model ({top_max_cap}) unchanged across class balance sweep."
    elif top_min_cap == primary_model:
        balance_flag = "FREQUENCY-DRIVEN"
        balance_note = f"Ranking flips from {top_max_cap} → {top_min_cap} at balanced classes — original ranking was frequency-driven."
    else:
        balance_flag = "INCONSISTENT"
        balance_note = f"Ranking flips from {top_max_cap} → {top_min_cap} at balanced classes."
else:
    balance_flag = "UNKNOWN"
    balance_note = f"No data for k={k_ref}, metric={primary_metric_label}."

print(f"\nStep 5 — Balance flag: {balance_flag}")
print(f"  {balance_note}")

#### 5.3b Label noise

`plot_knn_alignment_vs_feature_noise` progressively corrupts a fraction of labels while keeping the embedding geometry fixed. A score that degrades gracefully with noise indicates that the class structure is genuinely encoded in the embeddings, rather than being an artefact of a few perfectly labelled hubs.

In [ ]:
from emmaemb.visualization import plot_knn_alignment_vs_feature_noise

fig_noise, df_noise = plot_knn_alignment_vs_feature_noise(
    emma=emma,
    feature=feature,
    emb_spaces=models,
    k_values=k_values,
    metrics=distance_metrics,
    n_noise_steps=8,
    n_repeats=3,
    seed=42,
    show_random_baselines=True,
    return_data=True,
)
fig_noise.update_layout(height=500, width=900)
fig_noise.show()

# Step 6: noise robustness
metric_display = {"cosine": "Cosine distance", "cityblock": "Manhattan distance", "euclidean": "Euclidean distance"}
k_ref = k_values[len(k_values) // 2]
primary_metric_label = metric_display.get(distance_metrics[0], distance_metrics[0])

df_n = (
    df_noise[(df_noise["k"] == k_ref) & (df_noise["Metric"] == primary_metric_label)]
    .groupby(["Noise fraction", "Embedding"])["Fraction"]
    .mean()
    .unstack("Embedding")
)

if not df_n.empty and len(df_n.columns) > 1:
    leader_at_zero = df_n.iloc[0].idxmax()
    crossings = df_n[df_n.idxmax(axis=1) != leader_at_zero]
    if crossings.empty or float(crossings.index[0]) > 0.3:
        noise_flag = "ROBUST"
        crossing_frac = float(crossings.index[0]) if not crossings.empty else None
        noise_note = (
            f"Rankings stable up to noise={crossing_frac:.2f}."
            if crossing_frac is not None
            else "Rankings stable across all noise levels."
        )
    else:
        noise_flag = "SENSITIVE"
        crossing_frac = float(crossings.index[0])
        noise_note = f"Rankings flip at noise={crossing_frac:.2f} — class structure is noise-sensitive."
else:
    noise_flag = "ROBUST"
    noise_note = "Single model or no data — noise robustness not applicable."

print(f"\nStep 6 — Noise flag: {noise_flag}")
print(f"  {noise_note}")

# Overall confidence assessment
df_ref = df_knn[df_knn["k"] == k_ref].groupby(["distance_metric", "Embedding"])["Fraction"].mean()
margins = []
for metric_key in distance_metrics:
    if metric_key in df_ref.index.get_level_values("distance_metric"):
        grp = df_ref.xs(metric_key, level="distance_metric").sort_values(ascending=False)
        if len(grp) >= 2:
            margins.append(float(grp.iloc[0] - grp.iloc[1]))
margin = float(np.mean(margins)) if margins else 0.0

all_robust = stability_flag == "STABLE" and balance_flag == "CONSISTENT" and noise_flag == "ROBUST"
any_unreliable = (
    stability_flag == "PARAMETER-SENSITIVE"
    or balance_flag == "INCONSISTENT"
    or noise_flag == "SENSITIVE"
)

if all_robust and margin > 0.10 and rhi_max_ < 0.3:
    confidence = "HIGH"
elif any_unreliable or margin <= 0.05:
    confidence = "LOW"
else:
    confidence = "MODERATE"

print(f"\n=== Final Assessment ===")
print(f"  Stability:    {stability_flag}")
print(f"  Balance:      {balance_flag}")
print(f"  Noise:        {noise_flag}")
print(f"  Hubness:      {'HIGH' if rhi_max_ > 0.3 else 'acceptable'} (RHI_max={rhi_max_:.3f})")
print(f"  Margin:       {margin:.3f}")
print(f"  → Confidence: {confidence}")
print(f"  → Primary model: {primary_model}")

In [ ]:
from emmaemb.visualization import plot_knn_alignment_across_classes

fig_alignment_scores_class = plot_knn_alignment_across_classes(
    emma, feature="enzyme_class", k=100, metric="cityblock"
)
fig_alignment_scores_class.update_layout(height=600, width=500)
fig_alignment_scores_class.show()

### 4.2 KNN class mixing matrix

The KNN class mixing matrix quantifies the mixing of classes within the KNN neighborhood of samples in a given embedding space.
Given a distance metric (default: euclidean), the KNN are retrieved for each sample and the KNN class `get_class_mixing_in_neighborhood` counts how often different classes appear among its neighbors. The function returns a class mixing matrix, where each entry represents the number of times a class appears in the neighborhood of another class, along with the unique class labels. The heatmap can be visualised using the `plot_knn_class_mixing_matrix` function.

In [ ]:
# KNN CLASS MIXING MATRIX
from emmaemb.visualization import plot_knn_class_mixing_matrix

fig_class_mixing_matrix = plot_knn_class_mixing_matrix(
    emma,
    emb_space="ProtT5",
    feature="enzyme_class",
    k=100,
    metric="cityblock",
)
fig_class_mixing_matrix.update_layout(height=600, width=600)
fig_class_mixing_matrix.show()

<a name="section-five"></a>
## 5. Pairwise space comparison

### 5.1 Global comparison of pairwise distances

The `plot_pairwise_distance_comparison` function generates a scatter plot to compare pairwise distances between samples in two different embedding spaces. Using a specified distance metric (default: euclidean), it shows the distances for the same set of samples across both embedding spaces.
Additionally it computes the Spearman correlation coefficient between the pairwise distances in the two selected embedding spaces.
The function allows customization of plot title, color, and scatter dot opacity, and optionally groups points based on a meta data feature, enabling insights into how different sample categories behave across embeddings.

In [ ]:
from emmaemb.visualization import plot_pairwise_distance_comparison

fig_pwd_comparison = plot_pairwise_distance_comparison(
    emma,
    emb_space_y="ProtT5",
    emb_space_x="ESMC",
    metric="cityblock",
    group_by="species",
)
fig_pwd_comparison.update_layout(height=600, width=600)
fig_pwd_comparison.show()

### 5.2 Cross-space neighborhood similarity

The `plot_low_similarity_distribution` function visualizes the class distribution of samples with low neighborhood similarity between two embedding spaces. It computes neighborhood similarity scores based on a specified distance metric (default: euclidean) and identifies samples where similarity of the nearest neighbors of a data point falls below a given threshold. The function then compares the class distribution of these low-similarity samples to the overall dataset distribution using a scatter plot. This helps assess whether certain classes exhibit higher or lower structural consistency across embeddings, providing insights into differences in how embeddings capture relationships between samples.

In [ ]:
from emmaemb.visualization import plot_low_similarity_distribution

fig_low_similarity_class_distribution = plot_low_similarity_distribution(
    emma,
    emb_space_1="ProtT5",
    emb_space_2="ESMC",
    feature="enzyme_class",
    k=10,
    metric="cityblock",
    similarity_threshold=0.3,
)
fig_low_similarity_class_distribution.update_layout(height=600, width=600)
fig_low_similarity_class_distribution.show()